# 3장. 지저분한 줄 정리하고 검산하기

합친 표에는 빈 줄, 합계 줄, 글자로 적힌 날짜와 금액, 두 번 들어간 전표가 섞여 있다.
하나씩 눈으로 확인하고 고친 다음, 마지막에 함수 하나로 묶는다.

In [1]:
from pathlib import Path

import pandas as pd

WORK = Path("../work")
WEEK = "2026-11-02"

# 일단 전부 글자로 읽는다. 무엇을 숫자와 날짜로 바꿀지는 직접 정한다
merged = pd.read_csv(WORK / "02_merged.csv", dtype="string")
df = merged.copy()
print(len(df), "줄")
df.dtypes

259 줄


지점      string
원본파일    string
전표번호    string
판매일     string
품목코드    string
품목명     string
수량      string
단가      string
금액      string
dtype: object

## 1. 판매 내역이 아닌 줄 걸러 내기
전표번호가 `GN-1102-001` 모양이 아닌 줄을 먼저 본다.

In [2]:
SLIP_PATTERN = r"^[A-Z]{2}-\d{4}-\d{3}$"

slip = df["전표번호"].str.strip()
is_sale = slip.str.match(SLIP_PATTERN).fillna(False).astype(bool)

df[~is_sale]

,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액
84,광주점,광주점_주간판매.xlsx,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
100,광주점,광주점_주간판매.xlsx,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
258,수원점,수원점_주간판매.xlsx,합계,<NA>,<NA>,<NA>,515.0,<NA>,5975100


광주점의 빈 줄 둘과 수원점의 합계 줄이다. 합계 줄은 버리기 전에 따로 챙겨 둔다. 검산에 쓴다.

In [3]:
total_rows = df[slip == "합계"]
df = df[is_sale].copy()
df["전표번호"] = slip[is_sale]
print("남은 줄:", len(df))

남은 줄: 256


## 2. 글자 앞뒤 빈칸
눈에는 같아 보여도 `"물티슈 10팩"`과 `"물티슈 10팩  "`은 다른 값이다. 품목명 종류를 세어 보면 드러난다.

In [4]:
print("정리 전 품목명 종류:", df["품목명"].nunique())
for col in ["품목코드", "품목명"]:
    df[col] = df[col].str.strip()
print("정리 후 품목명 종류:", df["품목명"].nunique())

정리 전 품목명 종류: 17
정리 후 품목명 종류: 12


## 3. 날짜
지점마다 날짜가 어떻게 적혀 있는지 한 건씩 본다.

In [5]:
df.groupby("지점")["판매일"].first()

지점
강남점    2026-11-02 00:00:00
광주점    2026-11-02 00:00:00
대전점    2026-11-02 00:00:00
부산점             2026.11.02
수원점    2026-11-02 00:00:00
Name: 판매일, dtype: string

In [6]:
def to_date(series):
    """'2026.11.03' 같은 글자를 날짜로 바꾼다. 못 바꾼 칸은 빈 값이 된다."""
    text = series.astype("string").str.strip().str.replace(".", "-", regex=False)
    return pd.to_datetime(text, format="ISO8601", errors="coerce")


df["판매일"] = to_date(df["판매일"])
print("날짜로 못 바꾼 줄:", df["판매일"].isna().sum())
print(df["판매일"].min().date(), "~", df["판매일"].max().date())

날짜로 못 바꾼 줄: 0
2026-11-02 ~ 2026-11-07


## 4. 숫자
부산점 금액은 `"103,600원"`처럼 글자다. 쉼표, '원', 빈칸을 지우고 숫자로 바꾼다.

In [7]:
def to_number(series):
    """'1,250,000원' 같은 글자를 숫자로 바꾼다. 못 바꾼 칸은 빈 값이 된다."""
    text = series.astype("string").str.replace(r"[,원\s]", "", regex=True)
    return pd.to_numeric(text, errors="coerce")


for col in ["수량", "단가", "금액"]:
    df[col] = to_number(df[col])
print("숫자로 못 바꾼 칸:")
print(df[["수량", "단가", "금액"]].isna().sum())

숫자로 못 바꾼 칸:
수량    0
단가    0
금액    0
dtype: int64


In [8]:
for col in ["수량", "단가", "금액"]:
    df[col] = df[col].astype("int64")
df.dtypes

지점              string
원본파일            string
전표번호            string
판매일     datetime64[us]
품목코드            string
품목명             string
수량               int64
단가               int64
금액               int64
dtype: object

## 5. 두 번 들어간 전표
줄 전체가 같은지로 보지 않고, 같은 지점에 같은 전표번호가 또 있는지로 본다.

In [9]:
dup = df.duplicated(subset=["지점", "전표번호"], keep=False)
df[dup].sort_values("전표번호")

,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액
75,광주점,광주점_주간판매.xlsx,GJ-1103-008,2026-11-03,P-201,물티슈 10팩,2,11900,23800
76,광주점,광주점_주간판매.xlsx,GJ-1103-008,2026-11-03,P-201,물티슈 10팩,2,11900,23800
80,광주점,광주점_주간판매.xlsx,GJ-1103-012,2026-11-03,P-101,A4 복사용지 1박스,3,24500,73500
81,광주점,광주점_주간판매.xlsx,GJ-1103-012,2026-11-03,P-101,A4 복사용지 1박스,3,24500,73500
114,광주점,광주점_주간판매.xlsx,GJ-1107-043,2026-11-07,P-201,물티슈 10팩,20,11900,238000
115,광주점,광주점_주간판매.xlsx,GJ-1107-043,2026-11-07,P-201,물티슈 10팩,20,11900,238000


In [10]:
before = len(df)
df = df[~df.duplicated(subset=["지점", "전표번호"], keep="first")].reset_index(drop=True)
print(before, "→", len(df), "줄")

256 → 253 줄


## 6. 검산
수원점은 파일 맨 아래에 합계를 적어 보냈다. 우리가 계산한 수원점 합계와 같은지 본다.

In [11]:
theirs = int(to_number(total_rows["금액"]).iloc[0])
ours = int(df.loc[df["지점"] == "수원점", "금액"].sum())
print(f"파일에 적힌 합계 {theirs:,}원 / 계산한 합계 {ours:,}원 →", "일치" if theirs == ours else "불일치")

파일에 적힌 합계 5,975,100원 / 계산한 합계 5,975,100원 → 일치


금액이 수량×단가와 다른 줄, 날짜가 이번 주가 아닌 줄을 찾는다. 이런 줄은 고치지 않고 따로 모아 둔다.

In [12]:
monday = pd.Timestamp(WEEK)
wrong_amount = df["금액"] != df["수량"] * df["단가"]
out_of_week = ~df["판매일"].between(monday, monday + pd.Timedelta(days=6))

check = df[wrong_amount | out_of_week].copy()
check["사유"] = ""
check.loc[wrong_amount, "사유"] = "금액이 수량×단가와 다름"
check.loc[out_of_week, "사유"] = "판매일이 이번 주가 아님"
check

,지점,원본파일,전표번호,판매일,품목코드,품목명,수량,단가,금액,사유
177,부산점,부산점_주간판매.xlsx,BS-1105-026,2026-11-05,P-102,유성펜 12자루,2,6800,103600,금액이 수량×단가와 다름


In [13]:
df.to_csv(WORK / "03_clean.csv", index=False, encoding="utf-8-sig")
check.to_csv(WORK / "03_check.csv", index=False, encoding="utf-8-sig")
print(f"정리한 판매 {len(df)}건, 합계 {df['금액'].sum():,}원")

정리한 판매 253건, 합계 29,919,700원


## 실습. 한 함수로 묶어서 지난주에도 쓰기
위에서 한 일을 순서대로 넣은 함수다. 지난주 표에 돌려서 같은 정리가 되는지 본다.

In [14]:
def clean(merged, week):
    """합친 표를 정리한다. (정리한 표, 확인이 필요한 줄, 메모 목록)을 돌려준다."""
    notes = []
    df = merged.copy()

    # 1. 전표번호가 전표 모양인 줄만 판매 내역이다. 빈 줄과 합계 줄이 여기서 빠진다
    slip = df["전표번호"].astype("string").str.strip()
    is_sale = slip.str.match(SLIP_PATTERN).fillna(False).astype(bool)
    total_rows = df[slip == "합계"]
    df = df[is_sale].copy()
    df["전표번호"] = slip[is_sale]

    # 2. 글자 앞뒤 빈칸
    for col in ["품목코드", "품목명"]:
        df[col] = df[col].astype("string").str.strip()

    # 3. 날짜와 숫자
    df["판매일"] = to_date(df["판매일"])
    for col in ["수량", "단가", "금액"]:
        df[col] = to_number(df[col])
    broken = df[df[["판매일", "수량", "단가", "금액"]].isna().any(axis=1)]
    if len(broken):
        where = ", ".join(broken["전표번호"].head(5))
        raise ValueError(f"날짜나 숫자로 읽지 못한 줄이 {len(broken)}건 있습니다. 전표번호: {where}")
    for col in ["수량", "단가", "금액"]:
        df[col] = df[col].astype("int64")

    # 4. 같은 지점에 같은 전표번호가 두 번 있으면 한 번만 남긴다
    dup = df.duplicated(subset=["지점", "전표번호"], keep="first")
    if dup.any():
        counts = df[dup].groupby("지점").size()
        notes.append("중복 전표 제거: " + ", ".join(f"{b} {n}건" for b, n in counts.items()))
    df = df[~dup].reset_index(drop=True)

    # 5. 지점이 적어 보낸 합계와 맞는지 검산한다
    for _, row in total_rows.iterrows():
        branch = row["지점"]
        theirs = int(to_number(pd.Series([row["금액"]])).iloc[0])
        ours = int(df.loc[df["지점"] == branch, "금액"].sum())
        mark = "일치" if theirs == ours else "불일치, 확인 필요"
        notes.append(f"{branch} 합계 검산: 파일 {theirs:,}원 / 계산 {ours:,}원 → {mark}")

    # 6. 고치지 않고 표시만 하는 줄: 금액이 수량×단가와 다르거나, 날짜가 그 주가 아닌 줄
    monday = pd.Timestamp(week)
    wrong_amount = df["금액"] != df["수량"] * df["단가"]
    out_of_week = ~df["판매일"].between(monday, monday + pd.Timedelta(days=6))
    check = df[wrong_amount | out_of_week].copy()
    check["사유"] = ""
    check.loc[wrong_amount, "사유"] = "금액이 수량×단가와 다름"
    check.loc[out_of_week, "사유"] = "판매일이 이번 주가 아님"
    if len(check):
        notes.append(f"확인이 필요한 줄 {len(check)}건 ('확인필요' 시트)")

    return df, check, notes


# 손으로 한 결과와 함수 결과가 같은지 먼저 확인한다
again, _, _ = clean(merged, WEEK)
print("이번 주, 손으로 한 것과 같은가:", again["금액"].sum() == df["금액"].sum() and len(again) == len(df))

prev_merged = pd.read_csv(WORK / "02_merged_prev.csv", dtype="string")
prev, prev_check, prev_notes = clean(prev_merged, "2026-10-26")
prev.to_csv(WORK / "03_clean_prev.csv", index=False, encoding="utf-8-sig")
print(f"지난주 판매 {len(prev)}건, 합계 {prev['금액'].sum():,}원")
for note in prev_notes:
    print(" -", note)

이번 주, 손으로 한 것과 같은가: True
지난주 판매 252건, 합계 31,790,000원
 - 중복 전표 제거: 광주점 3건
 - 수원점 합계 검산: 파일 5,894,600원 / 계산 5,894,600원 → 일치
